# Modelos del lenguaje II (ahora es personal)

Un modelo del lenguaje permite estimar la probabilidad de una palabra dada una historia. En el caso de los bigramas, se asume la propiedad de Markov y se toma únicamente el elemento inmediatamente anterior. En el caso de modelos de n-gramas, con $n > 2$, la palabra depende de más elementos anteriores.

Por ejemplo, un modelo de 3-gramas toma en cuenta las dos palabras anteriores, un modelo de 4-gramas toma en cuenta tres palabras anteriores  y en general un modelo de n-gramas tomatá en cuenta $n-1$ palabras anteriores.

> Un modelo del lenguaje es un modelo estadístico que asigna probabilidades a cadenas dentro de un lenguaje - Jurafsky, 2000

$$ \mu = (\Sigma, A, \Pi)$$

Donde:
- $\mu$ es el modelo del lenguaje
- $\Sigma$ es el vocabulario
- $A$ es el tensor que guarda las probabilidades
- $\Pi$ guarda las probabilidades iniciales

### I saw a cat in a mat

<img src="https://lena-voita.github.io/resources/lectures/lang_models/general/i_saw_a_cat_prob.gif">

## Programemos un modelo de 3-gramas

In [1]:
import re
from collections import Counter, defaultdict
from itertools import chain

import pandas as pd
import numpy as np
from rich import print as rprint
from sklearn.model_selection import train_test_split

In [2]:
def vocab() -> defaultdict:
    """Crea diccionario de vocabulario

    Asigna a cada palabra un índice numerico
    único

    Return
    ------
    defaultdict
        Diccionario con el vocabulario
    """
    vocab = defaultdict()
    vocab.default_factory = lambda: len(vocab)
    return vocab

In [3]:
def text2number(corpus: list[str], vocab: defaultdict) -> list[int]:
    """Convierte una cadena de simbolos a una secuencia numerica

    Parameters
    ----------
    corpus: list
        Lista con oraciones
    vocab: defaultdict
        Diccionario que asigna indices únicos por cada palabra

    Return
    ------
    list[int]:
        Lista con indices de palabras
    """
    result = []
    for sent in corpus:
        result.append([vocab[word] for word in sent])
    return result

In [4]:
def get_invert_vocab(vocab: defaultdict) -> dict:
    return {idx: word for word, idx in vocab.items()}

def number2text(corpus: list[int], vocab: dict) -> list[str]:
    result = []
    for word_indices in corpus:
        result.append([vocab[idx] for idx in word_indices])
    return result

### Un nuevo corpus: CESS en Español

In [5]:
import nltk

In [6]:
nltk.download("cess_esp")

[nltk_data] Downloading package cess_esp to /root/nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


True

In [7]:
from nltk.corpus import cess_esp

raw_corpus = cess_esp.sents()[:100]
rprint(len(raw_corpus))

100

In [8]:
raw_corpus[:3]

[['El',
  'grupo',
  'estatal',
  'Electricité_de_France',
  '-Fpa-',
  'EDF',
  '-Fpt-',
  'anunció',
  'hoy',
  ',',
  'jueves',
  ',',
  'la',
  'compra',
  'del',
  '51_por_ciento',
  'de',
  'la',
  'empresa',
  'mexicana',
  'Electricidad_Águila_de_Altamira',
  '-Fpa-',
  'EAA',
  '-Fpt-',
  ',',
  'creada',
  'por',
  'el',
  'japonés',
  'Mitsubishi_Corporation',
  'para',
  'poner_en_marcha',
  'una',
  'central',
  'de',
  'gas',
  'de',
  '495',
  'megavatios',
  '.'],
 ['Una',
  'portavoz',
  'de',
  'EDF',
  'explicó',
  'a',
  'EFE',
  'que',
  'el',
  'proyecto',
  'para',
  'la',
  'construcción',
  'de',
  'Altamira_2',
  ',',
  'al',
  'norte',
  'de',
  'Tampico',
  ',',
  'prevé',
  'la',
  'utilización',
  'de',
  'gas',
  'natural',
  'como',
  'combustible',
  'principal',
  'en',
  'una',
  'central',
  'de',
  'ciclo',
  'combinado',
  'que',
  'debe',
  'empezar',
  'a',
  'funcionar',
  'en',
  'mayo_del_2002',
  '.'],
 ['La',
  'electricidad',
  'producida',

#### Preprocesamiento

In [8]:
def preprocess(sents: list[list[str]]) -> list[str]:
    return [[word.lower() for word in sent if re.match("[\w\s]", word)] for sent in sents]

In [10]:
rprint(preprocess(raw_corpus[-1:]))

[
    [
        'las',
        'chinelas',
        'de',
        'su',
        'madre',
        'sus',
        'tobillos',
        'siempre',
        'al',
        'aire',
        'sobre',
        'los',
        'tacones',
        'que',
        'según',
        'afirmaba',
        'con',
        'convicción',
        'eran',
        'indispensables',
        'para',
        'parecer',
        'arreglada',
        'atractiva',
        'incluso',
        'hasta',
        'en',
        'los',
        'peores',
        'momentos',
        'de',
        'la',
        'jornada',
        'doméstica',
        'le',
        'precedían',
        'por',
        'la',
        'estrecha',
        'escalera',
        'de',
        'la',
        'azotea',
        'iluminando',
        'para',
        'él',
        'los',
        'aterradores',
        'tramos',
        'que',
        'jamás',
        'habría',
        'sido',
        'capaz',
        'de',
        'coronar',
        'solo'
    ]
]

In [9]:
corpus = preprocess(raw_corpus)

In [11]:
rprint(len(corpus))

250

In [10]:
train_data, test_data = train_test_split(corpus, test_size=0.3)

In [14]:
rprint(len(train_data), len(test_data))

4221 1809

In [11]:
cess_vocab = vocab()
cess_indices = list(text2number(train_data, cess_vocab))

In [12]:
#Indicamos las etiquetas a usar
EOS = '<EOS>'
BOS = '<BOS>'

#Cada etiqeuta se le asigna un indice numerico
BOS_IDX = max(cess_vocab.values()) + 2
EOS_IDX = max(cess_vocab.values()) + 1

# Se agregan estas etiquetas al vocabulario
cess_vocab[EOS] = EOS_IDX
cess_vocab[BOS] = BOS_IDX

# A cada cadena se le agrega el índice de la etiqueta BOS al inicio y EOS al final
cess_indices = [[BOS_IDX] + idx_sent + [EOS_IDX] for idx_sent in cess_indices]

#Diccionario de índice : palabra
cess_words_idx = get_invert_vocab(cess_vocab)

In [13]:
for word, idx in cess_vocab.items():
    if idx >= 10:
        break
    rprint(word, idx)

1991 0

la_declaración_de_guadalajara 1

destaca 2

el 3

apoyo 4

a 5

los 6

procesos 7

de 8

integración 9

In [13]:
rprint(cess_indices[:3])
rprint(len(cess_indices))
rprint("EOS_IDX=", EOS_IDX, "BOS_IDX=", BOS_IDX)

[
    [
        913,
        0,
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9,
        10,
        11,
        1,
        2,
        12,
        13,
        14,
        15,
        16,
        13,
        17,
        18,
        19,
        1,
        7,
        20,
        21,
        22,
        23,
        24,
        25,
        7,
        26,
        20,
        27,
        28,
        29,
        7,
        30,
        31,
        32,
        1,
        7,
        33,
        34,
        18,
        35,
        36,
        912
    ],
    [
        913,
        37,
        38,
        39,
        13,
        40,
        41,
        42,
        10,
        43,
        44,
        1,
        45,
        46,
        1,
        45,
        20,
        47,
        48,
        49,
        50,
        51,
        7,
        52,
        53,
        54,
        55,
        56,
        57,
        912
    ],
    [913, 55, 58, 7, 59, 13, 60, 912]
]

70

EOS_IDX= 912 BOS_IDX= 913

In [14]:
rprint(number2text(cess_indices[:3], cess_words_idx))

[
    [
        '<BOS>',
        'además',
        'en',
        'las',
        'elecciones',
        'generales',
        'celebradas',
        'hoy',
        'el',
        'pp',
        'consolidó',
        'su',
        'entrada',
        'en',
        'las',
        'localidades',
        'del',
        'denominado',
        'cinturón',
        'rojo',
        'del',
        'sur',
        'de',
        'madrid',
        'en',
        'el',
        'que',
        'comparte',
        'victorias',
        'y',
        'derrotas',
        'con',
        'el',
        'psoe',
        'que',
        'sin_embargo',
        'sigue',
        'siendo',
        'el',
        'partido',
        'más',
        'votado',
        'en',
        'el',
        'cómputo',
        'global',
        'de',
        'esta',
        'zona',
        '<EOS>'
    ],
    [
        '<BOS>',
        'australia',
        'actual',
        'campeona',
        'del',
        'torneo',
        'intentará',
        'aprovechar',
        'su',
        'teórica',
        'ventaja',
        'en',
        'la',
        'hierba',
        'en',
        'la',
        'que',
        'los',
        'luchadores',
        'jugadores',
        'brasileños',
        'salvo',
        'el',
        'todoterreno',
        'gustavo_kuerten',
        'no',
        'se',
        'adaptan',
        'bien',
        '<EOS>'
    ],
    [
        '<BOS>',
        'se',
        'suscribió',
        'el',
        'programa',
        'del',
        'fondo_para_el_desarrollo_de_los_pueblos_indígenas',
        '<EOS>'
    ]
]

### Modelo de lenguaje


Una vez preprocesadas las cadenas pasaremos a estimar el modelo. Para esta estimación, tomaremos en cuenta dos parámetros:

*   El tamaño de n-gramas; es decir, qué tantos elementos previos consideraremos para estimar la probabilidad de que ocurra una palabra.
    - bigramas
    - trigramas
    - etc
*   El elemento $\lambda$ para estimar la probabilidad con smoothing de Lidstone. En ese sentido, dado un n-grama $w_{i-n+1} ... w_{i-1} w_i$ estimaremos la probabilidad como:

$$p(w_i|w_{i-1}...w_{i-n+1}) = \frac{C(w_{i-n+1} ... w_{i-1} w_i) + \lambda}{C(w_{i-n+1} ... w_{i-1}) + \lambda V}$$

donde $V$ es el tamaño del vocabulario.

In [15]:
def get_ngrams(sentences: list, n: int) -> list[tuple]:
    return chain(*[zip(*[sent[i:] for i in range(n)]) for sent in sentences])

In [21]:
rprint(list(get_ngrams(cess_indices[:3], n=3)))

[
    (19872, 0, 1),
    (0, 1, 2),
    (1, 2, 3),
    (2, 3, 4),
    (3, 4, 5),
    (4, 5, 6),
    (5, 6, 7),
    (6, 7, 8),
    (7, 8, 9),
    (8, 9, 10),
    (9, 10, 11),
    (10, 11, 12),
    (11, 12, 13),
    (12, 13, 14),
    (13, 14, 9),
    (14, 9, 15),
    (9, 15, 16),
    (15, 16, 17),
    (16, 17, 18),
    (17, 18, 19),
    (18, 19, 20),
    (19, 20, 21),
    (20, 21, 22),
    (21, 22, 23),
    (22, 23, 7),
    (23, 7, 24),
    (7, 24, 9),
    (24, 9, 25),
    (9, 25, 19871),
    (19872, 26, 27),
    (26, 27, 28),
    (27, 28, 29),
    (28, 29, 30),
    (29, 30, 28),
    (30, 28, 31),
    (28, 31, 32),
    (31, 32, 33),
    (32, 33, 34),
    (33, 34, 35),
    (34, 35, 28),
    (35, 28, 31),
    (28, 31, 36),
    (31, 36, 37),
    (36, 37, 7),
    (37, 7, 38),
    (7, 38, 9),
    (38, 9, 39),
    (9, 39, 40),
    (39, 40, 41),
    (40, 41, 42),
    (41, 42, 13),
    (42, 13, 43),
    (13, 43, 44),
    (43, 44, 7),
    (44, 7, 45),
    (7, 45, 19871),
    (19872, 46, 47),
    (46, 47, 31),
    (47, 31, 48),
    (31, 48, 49),
    (48, 49, 50),
    (49, 50, 51),
    (50, 51, 52),
    (51, 52, 53),
    (52, 53, 54),
    (53, 54, 55),
    (54, 55, 20),
    (55, 20, 56),
    (20, 56, 57),
    (56, 57, 37),
    (57, 37, 13),
    (37, 13, 58),
    (13, 58, 9),
    (58, 9, 13),
    (9, 13, 59),
    (13, 59, 60),
    (59, 60, 61),
    (60, 61, 62),
    (61, 62, 23),
    (62, 23, 4),
    (23, 4, 63),
    (4, 63, 19871)
]

In [16]:
def get_model(corpus: list, vocab: defaultdict, n: int=2, l: float=1.0):
    ngrams = get_ngrams(corpus, n)

    freq_grams = Counter(ngrams)

    # Obtenermos el tamaño del vocabulario
    # Quitamos el EOS y BOS
    V = len(vocab) - 2

    # Calculo de la dimensión del tensor de transiciones
    # En palabras condicionadas consideraremos al elemento EOS
    dim = (V,) * (n - 1) + (V + 1,)
    # Tensor de transiciones
    A = np.zeros(dim)
    # Probabilidades iniciales
    Pi = np.zeros(V)

    # Calculo de frecuencias
    for ngram, freq in freq_grams.items():
        # Llenado del tensor de transiciones
        if ngram[0] != BOS_IDX:
            A[ngram] = freq
        # Llenado de frecuencias iniciales
        elif ngram[0] == BOS_IDX and ngram[1] != EOS_IDX:
            Pi[ngram[1]] = freq

    # Calculo de probabilidades a partir de frecuencias
    # El parámetro l es para smoothing
    for i, b in enumerate(A):
        A[i] = (
            (b + l).T /
            (b + l).sum(n - 2)
        ).T

    # Calculo de probabilidades iniciales
    Pi = (Pi + l) / (Pi + l).sum(0)

    return A, Pi

In [17]:
%%time
trigram_cess = get_model(cess_indices, cess_vocab, n=3, l=1.0)

CPU times: user 4.84 s, sys: 11 s, total: 15.8 s
Wall time: 16 s


In [18]:
A_tri, Pi_tri = trigram_cess

In [19]:
rprint(A_tri.shape)
rprint(A_tri.sum(2))

(912, 912, 913)

[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]]

In [20]:
rprint(
    A_tri[
        cess_vocab["el"],
        cess_vocab["español"],
        cess_vocab["es"]
        ]
    )

0.001095290251916758

In [21]:
%%time
bigram_cess = get_model(cess_indices, cess_vocab, n=2, l=1)

CPU times: user 21.9 ms, sys: 653 µs, total: 22.5 ms
Wall time: 55.6 ms


In [22]:
A_bi, Pi_bi = bigram_cess

In [23]:
rprint(A_bi.shape)
rprint(A_bi.sum(1))

(912, 913)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

###  Obteniendo la probabilidad de una cadena

Para determinar la probabilidad, utilizaremos la función:

$$p(w_1 ... w_k) = \prod_{i=1}^k p(w_i|w_{i-1} ... w_{i-n+1})$$

Dado que las cadenas pueden extenderse y las probabilidades son pequeñas, es posible que la probabilidad se haga tan pequeña que aparezca como un cero. Para evitar esto, utilizaremos probabilidad logarítimicada, dada por:

$$\log p(w_1 ... w_k) = \sum_{i=1}^k \log p(w_i|w_{i-1} ... w_{i-n+1})$$

In [24]:
def get_sent_probability(sentence: list[str], vocab: defaultdict, model: tuple, verbose=False) -> float:
    A, Pi = model
    # Getting the n from n-grams
    n = len(A.shape)
    indexed_sentence = [vocab[word] for word in sentence]
    first_indexed_word = indexed_sentence[0]
    # Getting initial probability
    try:
        probability = np.log(Pi[first_indexed_word])
    except:
        if verbose:
            print(f"[WARN] OOV for word as BOS with index={first_indexed_word}")
        probability = 0.0

    # Getting n-grams of the sentence
    n_grams = get_ngrams([indexed_sentence], n)
    for n_gram in n_grams:
        try:
          probability += np.log(A[n_gram])
        except:
          if verbose:
            print(f"[WARN] OOV for n_gram={n_gram}")
          probability += 0.0

    return probability

In [34]:
from random import randint

sent = train_data[randint(0, len(train_data) - 1)]
rprint(f"Probabilidad de la cadena:\n [yellow]{' '.join(sent)}")
rprint(f"\t Modelo de trigramas: ",
       np.exp(get_sent_probability(sent, cess_vocab, trigram_cess)))
rprint(f"\t Modelo de bigramas: ",
       np.exp(get_sent_probability(sent, cess_vocab, bigram_cess)))

Probabilidad de la cadena:
 1993 bajo el lema un programa para el desarrollo el texto de la declaración resaltó la necesidad de impulsar el 
diálogo norte-sur acabar con el proteccionismo y dar prioridad a las negociaciones del 
acuerdo_general_sobre_aranceles_aduaneros_y_comercio gatt así_como conjugar el desarrollo social con el progreso 
económico

Modelo de trigramas:  8.536196510433796e-115

Modelo de bigramas:  1.621586731936501e-113

## Evaluación de modelos

La evaluación de  modelos del lenguaje se puede realizar a partir de su entropía. Esta se determina en un corpus de evaluación, que **nunca debió ser visto por el entrenamiento**. Calcularemos la entropía como:

$$H(p) = -\frac{1}{K} \sum_{i=1}^k \log p(w_1 ... w_k)$$

A partir de la entropía podríamos calcular la perplejidad como $2^{H(p)}$.

> Debe notarse que, como estamos estimando probabilidades logarítmicas, no hará falta obtener un logaritmo en esta función.

In [35]:
def get_entropy(model, vocab, test_data: list[list[str]]):
    # Inicialización Entropía
    H = 0.0
    # Evaluamos en el corpus de evaluación
    for sent in test_data:
        # Probabilidad de la cadena
        logp_cad = get_sent_probability(sent, vocab, model)
        # Número de palabras
        M = len(sent)
        # Obtenemos la entropía cruzada de la cadena
        H -= logp_cad / (M + 1e-100)

    return H / len(test_data)

In [36]:
rprint('--Perplejida--')
rprint('\t Modelo de bigramas:', 2**get_entropy(bigram_cess, cess_vocab, test_data))
rprint('\t Modelo de trigramas:', 2**get_entropy(trigram_cess, cess_vocab, test_data))

--Perplejida--

Modelo de bigramas: 5.8152650743253975

Modelo de trigramas: 3.132454825804735

En general, los modelos de trigramas muestran una entropía más baja (o perplejidad más baja) (Jurafsky y Martin, 2023).

## Otra forma de estimar modelos del lenguaje

![](https://imgs.xkcd.com/comics/predictive_models_2x.png)

In [37]:
new_corpus = cess_esp.sents()

In [38]:
train_data, test_data = train_test_split(new_corpus, train_size=0.7)

In [39]:
def preprocess_sent(sent: list[str]) -> list[str]:
    """Función de preprocesamiento

    Agrega tokens de inicio y fin a la oración y normaliza todo a minusculas

    Params
    ------
    sent: list[str]
        Lista de palabras que componen la oración

    Return
    ------
    Las oración preprocesada
    """
    result = [word.lower() for word in sent]
    # Al final de la oración
    result.append(EOS)
    # Al inicio de la oración
    result.insert(0, BOS)
    return result

In [40]:
rprint(preprocess_sent(train_data[-90]))

[
    '<BOS>',
    'cualquiera',
    'que',
    'lo',
    'hiciese',
    'con',
    'más',
    'jugadores',
    'relevándolos',
    'habitualmente',
    'era',
    'considerado',
    'como',
    'un',
    'loco',
    ',',
    'o',
    'poco',
    'menos',
    '.',
    '<EOS>'
]

In [41]:
from nltk import ngrams

rprint(list(ngrams(preprocess_sent(train_data[0]), n=3)))

[
    ('<BOS>', 'más_de', '1.200'),
    ('más_de', '1.200', 'candidatos'),
    ('1.200', 'candidatos', 'se'),
    ('candidatos', 'se', 'presentarán'),
    ('se', 'presentarán', 'a'),
    ('presentarán', 'a', 'las'),
    ('a', 'las', 'elecciones'),
    ('las', 'elecciones', ','),
    ('elecciones', ',', 'cuya'),
    (',', 'cuya', 'celebración'),
    ('cuya', 'celebración', 'coincidirá'),
    ('celebración', 'coincidirá', 'con'),
    ('coincidirá', 'con', 'los'),
    ('con', 'los', '63'),
    ('los', '63', 'años'),
    ('63', 'años', 'de'),
    ('años', 'de', 'edad'),
    ('de', 'edad', 'que'),
    ('edad', 'que', 'cumpliría'),
    ('que', 'cumpliría', 'el'),
    ('cumpliría', 'el', 'ex'),
    ('el', 'ex', 'primer'),
    ('ex', 'primer', 'ministro'),
    ('primer', 'ministro', 'keizo_obuchi'),
    ('ministro', 'keizo_obuchi', ','),
    ('keizo_obuchi', ',', 'fallecido'),
    (',', 'fallecido', 'el'),
    ('fallecido', 'el', '14_de_mayo'),
    ('el', '14_de_mayo', 'víctima'),
    ('14_de_mayo', 'víctima', 'de'),
    ('víctima', 'de', 'una'),
    ('de', 'una', 'hemorragia'),
    ('una', 'hemorragia', 'cerebral'),
    ('hemorragia', 'cerebral', '.'),
    ('cerebral', '.', '<EOS>')
]

In [42]:
test = defaultdict(Counter)

In [43]:
test[("hola", "que")]["hace"] += 1

In [44]:
test[("hola", "que")]

Counter({'hace': 1})

In [45]:
def build_ngram_model(data: list[list[str]], n: int) -> defaultdict:
    model = defaultdict(Counter)
    for sentence in data:
        for ngram in ngrams(preprocess_sent(sentence), n):
            if n == 2:
                context, current_word = ngram
            else:
                context = ngram[:-1]
                current_word = ngram[-1]
            model[context][current_word] += 1
    return model

In [46]:
%%time
trigram_model = build_ngram_model(train_data, n=3)

CPU times: user 732 ms, sys: 34.4 ms, total: 767 ms
Wall time: 771 ms


In [47]:
%%time
bigram_model = build_ngram_model(train_data, n=2)

CPU times: user 151 ms, sys: 7.16 ms, total: 158 ms
Wall time: 158 ms


In [48]:
trigram_model[BOS, "la"]

Counter({'coalición': 2,
         'clave': 3,
         'familia': 2,
         'duración': 1,
         'organización': 2,
         'octava': 1,
         'casi': 1,
         'concejala': 1,
         'medicina': 1,
         'lista': 1,
         'presencia': 3,
         'nueva': 3,
         'nieve': 1,
         'falta': 2,
         'sede': 1,
         'evolución': 2,
         'inteligencia': 1,
         'protesta': 1,
         'empresa': 5,
         'etarra': 1,
         'productora': 2,
         'cultura': 1,
         'policía': 5,
         'fiebre': 1,
         'corriente': 1,
         'producción': 1,
         'decisión': 7,
         'más': 1,
         'representante_comercial': 1,
         ',': 1,
         'tabla': 1,
         'asociación_para_la_paz': 1,
         'idea': 2,
         'queja': 1,
         'centralita': 1,
         'bandera': 2,
         'gaceta': 1,
         'rata': 2,
         'condición': 1,
         'selección': 3,
         'campaña': 2,
         'legislación': 2,
  

In [49]:
bigram_model[BOS]

Counter({'más_de': 2,
         '*0*': 302,
         'el': 546,
         'rominger': 5,
         'se': 26,
         'y': 78,
         'hace': 11,
         '-': 145,
         'muchas': 3,
         'según': 44,
         'medardo_fraile': 2,
         'para': 35,
         'henry_ford': 1,
         'los': 152,
         'uno': 7,
         'no_obstante': 6,
         'blanco': 1,
         'oribe': 1,
         'tales': 1,
         'no': 29,
         'pero': 88,
         'pese_a': 4,
         'un': 36,
         'djalminha': 1,
         'así_y_todo': 1,
         'tanto': 2,
         'reciclaje': 1,
         'hurtado': 1,
         'es': 18,
         'fuentes': 9,
         'de': 37,
         'desde': 27,
         'la': 321,
         'lo': 29,
         'precisamente': 4,
         'en': 195,
         'ella': 5,
         'guterres': 1,
         'al': 18,
         'su': 23,
         '1995': 1,
         '"': 170,
         'cecilia_calderón': 1,
         'pujol': 1,
         'pérez': 1,
         'por_otro

In [50]:
def get_vocab_size(corpus):
    return len(set([word.lower() for sent in corpus for word in sent])) + 2

def calculate_model_probs(model: defaultdict[Counter], corpus: list, l: int) -> defaultdict[Counter]:
    VOCABULARY_SIZE = get_vocab_size(corpus)
    model_probs = defaultdict(Counter)
    # Por cada prefijo del modelo
    for prefix in model:
        # Todas las veces que ocurre prefix seguido de cualquier palabra
        total=float(sum(model[prefix].values()))
        # Por cada palabra w que haya ocurrido con prefix
        for w in model[prefix]:
            # Obtenemos la probabilidad con smoothing
            model_probs[prefix][w] = (model[prefix][w] + l) / (total + (l * VOCABULARY_SIZE))
    return model_probs

In [51]:
%%time
bigram_probs = calculate_model_probs(bigram_model, new_corpus, l=1.0)

CPU times: user 4.69 s, sys: 302 ms, total: 4.99 s
Wall time: 5.08 s


In [52]:
%%time
trigram_probs = calculate_model_probs(trigram_model, new_corpus, l=1.0)

CPU times: user 4.06 s, sys: 251 ms, total: 4.31 s
Wall time: 4.32 s


In [56]:
rprint(sorted(dict(trigram_probs[BOS, "sin_embargo"]).items(), key=lambda x: -1*x[1]))

[
    (',', 0.0012229423994129876),
    ('serán', 8.152949329419917e-05),
    ('y', 8.152949329419917e-05),
    ('el', 8.152949329419917e-05)
]

In [55]:
rprint(sorted(dict(bigram_probs[BOS]).items(), key=lambda x: -1*x[1]))

[
    ('el', 0.019045961002785515),
    ('la', 0.011211699164345404),
    ('*0*', 0.010550139275766016),
    ('en', 0.006824512534818941),
    ('"', 0.005954038997214485),
    ('los', 0.005327298050139276),
    ('-', 0.005083565459610028),
    ('pero', 0.0030988857938718663),
    ('y', 0.0027506963788300836),
    ('por', 0.0017409470752089136),
    ('las', 0.001671309192200557),
    ('según', 0.0015668523676880223),
    ('una', 0.0014275766016713092),
    ('con', 0.0013579387186629527),
    ('de', 0.0013231197771587744),
    ('a', 0.0013231197771587744),
    ('un', 0.0012883008356545961),
    ('para', 0.0012534818941504179),
    ('sin_embargo', 0.001149025069637883),
    ('no', 0.0010445682451253482),
    ('lo', 0.0010445682451253482),
    ('desde', 0.0009749303621169916),
    ('se', 0.0009401114206128133),
    ('como', 0.0008704735376044568),
    ('este', 0.0008704735376044568),
    ('su', 0.0008356545961002785),
    ('hasta', 0.000766016713091922),
    ('además', 0.000766016713091922),
    ('tras', 0.0007311977715877437),
    ('entre', 0.0006963788300835655),
    ('cuando', 0.0006963788300835655),
    ('es', 0.0006615598885793872),
    ('al', 0.0006615598885793872),
    ('esta', 0.0006267409470752089),
    ('si', 0.0006267409470752089),
    ('¿', 0.0005222841225626741),
    ('también', 0.0005222841225626741),
    ('así', 0.0004874651810584958),
    ('hay', 0.0004874651810584958),
    ('que', 0.0004874651810584958),
    ('ahora', 0.00045264623955431753),
    ('después', 0.00045264623955431753),
    ('durante', 0.00045264623955431753),
    ('aunque', 0.00045264623955431753),
    ('hace', 0.00041782729805013927),
    ('mientras', 0.00041782729805013927),
    ('más', 0.00041782729805013927),
    ('porque', 0.00041782729805013927),
    ('yo', 0.000383008356545961),
    ('fuentes', 0.00034818941504178273),
    ('entonces', 0.00034818941504178273),
    ('ante', 0.00034818941504178273),
    ('todos', 0.00034818941504178273),
    ('esto', 0.00034818941504178273),
    ('todo', 0.00034818941504178273),
    ('.', 0.00031337047353760446),
    ('estos', 0.00031337047353760446),
    ('uno', 0.0002785515320334262),
    ('en_cuanto_a', 0.0002785515320334262),
    ('algunos', 0.0002785515320334262),
    ('brasil', 0.0002785515320334262),
    ('asimismo', 0.0002785515320334262),
    ('antes', 0.0002785515320334262),
    ('por_último', 0.0002785515320334262),
    ('sobre', 0.0002785515320334262),
    ('dos', 0.0002785515320334262),
    ('de_hecho', 0.0002785515320334262),
    ('ese', 0.0002785515320334262),
    ('no_obstante', 0.0002437325905292479),
    ('esa', 0.0002437325905292479),
    ('-fe-', 0.0002437325905292479),
    ('por_ejemplo', 0.0002437325905292479),
    ('chávez', 0.0002437325905292479),
    ('ambos', 0.0002437325905292479),
    ('ya', 0.0002437325905292479),
    ('todas', 0.0002437325905292479),
    ('estas', 0.0002437325905292479),
    ('sin', 0.0002437325905292479),
    ('a_partir_de', 0.0002437325905292479),
    ('otro', 0.0002437325905292479),
    ('rominger', 0.00020891364902506963),
    ('ella', 0.00020891364902506963),
    ('por_otro_lado', 0.00020891364902506963),
    ('germán_de_granda', 0.00020891364902506963),
    ('luego', 0.00020891364902506963),
    ('otros', 0.00020891364902506963),
    ('ni', 0.00020891364902506963),
    ('pese_a_que', 0.00020891364902506963),
    ('cada', 0.00020891364902506963),
    ('sólo', 0.00020891364902506963),
    ('pues', 0.00020891364902506963),
    ('incidencias', 0.00020891364902506963),
    ('ibarretxe', 0.00020891364902506963),
    ('unos', 0.00020891364902506963),
    ('quizá', 0.00020891364902506963),
    ('hoy', 0.00020891364902506963),
    ('sus', 0.00020891364902506963),
    ('tampoco', 0.00020891364902506963),
    ('le', 0.00020891364902506963),
    ('pese_a', 0.00017409470752089137),
    ('precisamente', 0.00017409470752089137),
    ('nada', 0.00017409470752089137),
    ('otras', 0.00017409470752089137),
    ('clinton', 0.00017409470752089137),
    ('allí', 0.000174094707520

### Aplicaciones

- Speech To Text (STT)
- Completado de texto
- Generación de texto

![](https://lena-voita.github.io/resources/lectures/lang_models/examples/suggest-min.png)
Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

#### Ejercicio: Utilizando el modelo de trigramas, diseña una estrategia para generación del lenguaje

In [57]:
def get_likely_words(
        model_probs: defaultdict[Counter],
        context: str,
        top_count: int=10) -> list[tuple]:
    """Dado un contexto obtiene las palabras más probables

    Params
    ------
    model_probs: defaultdict
        Probabilidades del modelo
    context: str
        Contexto con el cual calcular las palabras más probables siguientes
    top_count: int
        Cantidad de palabras más probables. Default 10
    """
    history = tuple(context.split())
    return sorted(dict(trigram_probs[history]).items(), key=lambda x: -1*x[1])[:top_count]

In [59]:
rprint(get_likely_words(trigram_probs, f"{BOS} el", top_count=2))

[('presidente', 0.0013575563984827312), ('gobierno', 0.0004791375524056698)]

In [95]:
rprint(get_likely_words(bigram_probs, "empresa", top_count=5))

[
    (',', 0.00032597180343900254),
    ('de', 0.00016298590171950127),
    ('por', 0.00012223942628962595),
    ('.', 0.00012223942628962595),
    ('que', 0.00012223942628962595)
]

In [ ]:
randint

In [74]:
from random import randint
def get_next_word(words: list) -> str:
    return words[randint(0, len(words) - 1)][0]

In [89]:
get_next_word(get_likely_words(trigram_probs, f"{BOS} el", top_count=50))

'portavoz'

In [99]:
import random
import time
MAX_TOKENS = 100
def generate_text(
        model: defaultdict[Counter], history: str, tokens_count: int = 0, top_n: int = 50
) -> None:
    next_word = get_next_word(get_likely_words(model, history, top_count=top_n))
    # Simulates high compute :p
    time.sleep(random.random())
    tokens_count += 1
    print(next_word, end=" ")
    if tokens_count == MAX_TOKENS or next_word == EOS:
        return
    generate_text(model, history.split()[1] + " " + next_word, tokens_count, top_n=top_n)

In [101]:
sent = "la empresa"
print(sent, end=" ")
generate_text(trigram_probs, sent)

la empresa mexicana electricidad_águila_de_altamira -fpa- eaa -fpt- , que podría entrar en el equipo de jóvenes investigadores y estudiantes que dirige el colombiano juan_pablo_angel , en un auténtico campeón . <EOS> 

### Calculando la probabilidad de una oración

In [102]:
def calculate_sentence_prob(sentence: list[str], model: defaultdict[Counter], n: int) -> float:
    prob = 0
    for ngram in ngrams(preprocess_sent(sentence), n):
        if n == 2:
            context, current_word = ngram
        else:
            context = ngram[:-1]
            current_word = ngram[-1]
        current_prob = model[context][current_word]
        if current_prob == 0:
            # OOV
            prob += 0.0
        else:
            prob += np.log(current_prob)
    return prob

In [104]:
for i, sent in enumerate(test_data):
    prob = calculate_sentence_prob(
        sent,
        trigram_probs,
        n=3
    )
    rprint(f"sent='{' '.join(sent)}'")
    rprint(f"P(sent)={np.exp(prob)}")
    if i == 2:
        break

sent='En su opinión , ante la " situación de transición " que ha generado la dimisión de Joaquín_Almunia y la 
Ejecutiva , la reunión del Comité_Federal , máximo órgano entre congresos , debe servir para que , aunque haya un 
debate muy abierto , el partido " salga unido " .'

P(sent)=8.607842009125776e-45

sent='¿ Aznar ?'

P(sent)=1.0

sent='- - Encarna - - *0* musitó y *0* se echó a llorar definitivamente , como si *0* hubiera asumido de_repente 
estar perdido en una ciudad sumergida .'

P(sent)=2.5074596210934436e-24